# 05 - Combined compression

###  1° Parte:  Combinazione di tecniche per Student

Student distillato → BN folding 
→ pruning strutturato con config di student_struct.json 
→ W8A8
- con confronto finale contro le due baseline

In [1]:
import sys; sys.path.append("..")
%load_ext autoreload
%autoreload 2

import json, os
import torch, torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
from src.data import get_dataloaders
from src.models import build_model
from src.quant import fold_bn, build_w8a8_model, int8_model_size_mb
from src.prune import prune_structured, skip_first_ratios
from src.train import evaluate
from src.metrics import profile_model, save_result, count_parameters

device = torch.device("cpu")
train_loader, val_loader, test_loader = get_dataloaders("../data", batch_size=128)

# config di pruning: la migliore trovata nel notebook 03 per lo student
cfg = json.load(open("../results/student_struct.json"))
CONV_RATIO, FC_RATIO, SKIP_FIRST = cfg["conv_ratio"], cfg["fc_ratio"], cfg["skip_first"]
print(f"pruning: conv {CONV_RATIO}, fc {FC_RATIO}, skip_first {SKIP_FIRST}")

student = build_model("student")
student.load_state_dict(torch.load("../checkpoints/student_kd.pt", map_location="cpu"))
folded = fold_bn(student)    # student distillato con BN fuso

pruning: conv 0.0, fc 0.3, skip_first 2


*  Pruning

In [2]:
conv_r = skip_first_ratios(folded, CONV_RATIO, SKIP_FIRST)
pruned = prune_structured(folded, conv_r, FC_RATIO)

_, val_kd = evaluate(folded, val_loader, device)
_, val_pr = evaluate(pruned, val_loader, device)
print(f"val: distillato {val_kd * 100:.2f} -> + pruning {val_pr * 100:.2f} (Δ {(val_pr - val_kd) * 100:+.2f})")
print("params:", count_parameters(folded)["total"], "->", count_parameters(pruned)["total"])

c:\Users\Pietr\Desktop\Model-Compression\.venv\lib\site-packages\torch\utils\data\dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


val: distillato 75.38 -> + pruning 74.96 (Δ -0.42)
params: 110021 -> 93457


* int8 W8A8 (stessa calibrazione degli altri notebook)

In [3]:
torch.manual_seed(0)
qmodel, qparams, bias_q = build_w8a8_model(pruned, train_loader, n_calib_batches=4)
_, val_q = evaluate(qmodel, val_loader, device)
print(f"val: + int8 W8A8 {val_q * 100:.2f} (Δ {(val_q - val_pr) * 100:+.2f})")

val: + int8 W8A8 74.92 (Δ -0.04)


In [5]:
# Test, profilling e salvataggio
_, test_pr = evaluate(pruned, test_loader, device)
_, test_q = evaluate(qmodel, test_loader, device)

prof_pr = profile_model(pruned, dtype="fp32")
save_result("student_kd_struct", val_acc=val_pr, test_acc=test_pr, conv_ratio=CONV_RATIO,
            fc_ratio=FC_RATIO, skip_first=SKIP_FIRST, **prof_pr)

prof_q = profile_model(qmodel, dtype="int8", latency=False,
                       size_mb=int8_model_size_mb(qparams, bias_q, len(qmodel.quantizers())))
save_result("student_kd_struct_int8", val_acc=val_q, test_acc=test_q, conv_ratio=CONV_RATIO,
            fc_ratio=FC_RATIO, skip_first=SKIP_FIRST, **prof_q)

print(f"test: pruning {test_pr * 100:.2f} | pruning + int8 {test_q * 100:.2f}")

test: pruning 75.48 | pruning + int8 75.25


Confronto finale

In [6]:
names = ["teacher_baseline", "student_baseline", "student_int8", "student_struct",
         "student_kd", "student_kd_struct", "student_kd_struct_int8"]
rows = {n: json.load(open(f"../results/{n}.json")) for n in names if os.path.exists(f"../results/{n}.json")}
df = pd.DataFrame(rows).T.drop(columns=["name"], errors="ignore").apply(pd.to_numeric, errors="coerce")

t, s = df.loc["teacher_baseline"], df.loc["student_baseline"]
table = pd.DataFrame({
    "test acc (%)": (df["test_acc"] * 100).round(2),
    "Δ vs teacher (pt)": ((df["test_acc"] - t["test_acc"]) * 100).round(2),
    "Δ vs student base (pt)": ((df["test_acc"] - s["test_acc"]) * 100).round(2),
    "params": df["params"].astype(int),
    "size (MB)": df["size_mb"].round(3),
    "size ÷ teacher": (t["size_mb"] / df["size_mb"]).round(1),
    "MACs (M)": (df["macs"] / 1e6).round(2),
    "peak RAM (KB)": df["peak_act_kb"].round(1),
    "energy (µJ)": df["energy_uj"].round(2),
    "energy ÷ teacher": (t["energy_uj"] / df["energy_uj"]).round(0),
    "latency (ms)": df["latency_ms_median"].round(2),   # CPU, 1 thread; vuota per le righe int8
})
display(table.style.format(precision=2, na_rep="–"))

# contributo di ogni stadio
chain = ["student_baseline", "student_kd", "student_kd_struct", "student_kd_struct_int8"]
for a, b in zip(chain, chain[1:]):
    if a in df.index and b in df.index:
        print(f"{a:>24} -> {b:<24}: {(df.loc[b, 'test_acc'] - df.loc[a, 'test_acc']) * 100:+.2f} pt")

,test acc (%),Δ vs teacher (pt),Δ vs student base (pt),params,size (MB),size ÷ teacher,MACs (M),peak RAM (KB),energy (µJ),energy ÷ teacher,latency (ms)
teacher_baseline,89.28,0.00,13.76,15530762,62.16,1.00,215.69,327.70,992.17,1.00,8.30
student_baseline,75.52,-13.76,0.00,110481,0.46,134.40,1.63,32.80,7.48,133.00,0.72
student_int8,75.46,-13.82,-0.06,110021,0.11,547.50,1.63,8.20,0.49,2034.00,–
student_struct,74.41,-14.87,-1.11,93457,0.38,163.00,1.61,32.80,7.40,134.00,0.54
student_kd,75.63,-13.65,0.11,110481,0.46,134.40,1.63,32.80,7.48,133.00,0.59
student_kd_struct,75.48,-13.80,-0.04,93457,0.38,163.00,1.61,32.80,7.40,134.00,0.52
student_kd_struct_int8,75.25,-14.03,-0.27,93457,0.10,644.60,1.61,8.20,0.48,2055.00,–


        student_baseline -> student_kd              : +0.11 pt
              student_kd -> student_kd_struct       : -0.15 pt
       student_kd_struct -> student_kd_struct_int8  : -0.23 pt


###  2° Parte:  Combinazione di tecniche per Teacher

→ pruning strutturato con config di teacher_struct.json 
→ W8A8
- con confronto finale contro la sua baseline

* Pruning

In [10]:
# Config di pruning: migliore configurazione trovata nel notebook 03
cfg = json.load(open("../results/teacher_struct.json"))

CONV_RATIO = cfg["conv_ratio"]
FC_RATIO = cfg["fc_ratio"]
SKIP_FIRST = cfg["skip_first"]

print(f"Teacher pruning: conv {CONV_RATIO}, fc {FC_RATIO}, skip_first {SKIP_FIRST}")


# Carico il teacher originale
teacher = build_model("teacher")
teacher.load_state_dict(
    torch.load("../checkpoints/teacher_baseline.pt", map_location="cpu")
)

folded = fold_bn(teacher)
# Applicazione pruning strutturato
conv_r = skip_first_ratios(folded, CONV_RATIO, SKIP_FIRST)
pruned = prune_structured(folded, conv_r, FC_RATIO)

_, val_base = evaluate(folded, val_loader, device)
_, val_pr = evaluate(pruned, val_loader, device)

print(f"Validation accuracy:")
print(f"  Teacher baseline: {val_base * 100:.2f}%")
print(f"  Teacher pruning:  {val_pr * 100:.2f}%")
print(f"  Δ: {(val_pr - val_base) * 100:+.2f} punti")

print("\nParametri:")
print(f"  Baseline: {count_parameters(folded)['total']:,}")
print(f"  Pruned:   {count_parameters(pruned)['total']:,}")

Teacher pruning: conv 0.0, fc 0.8, skip_first 2


c:\Users\Pietr\Desktop\Model-Compression\.venv\lib\site-packages\torch\utils\data\dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Validation accuracy:
  Teacher baseline: 89.84%
  Teacher pruning:  88.96%
  Δ: -0.88 punti

Parametri:
  Baseline: 15,525,258
  Pruned:   10,146,885


* W8A8 quantization

In [11]:
torch.manual_seed(0)

qmodel, qparams, bias_q = build_w8a8_model(
    pruned, train_loader, n_calib_batches=4
)

_, val_q = evaluate(qmodel, val_loader, device)

print(f"Validation accuracy:")
print(f"  Teacher pruning:       {val_pr * 100:.2f}%")
print(f"  Teacher pruning + INT8: {val_q * 100:.2f}%")
print(f"  Δ quantizzazione: {(val_q - val_pr) * 100:+.2f} punti")

Validation accuracy:
  Teacher pruning:       88.96%
  Teacher pruning + INT8: 88.90%
  Δ quantizzazione: -0.06 punti


Test profiling e salvataggio

In [14]:
# Test finale
_, test_base = evaluate(folded, test_loader, device)
_, test_pr = evaluate(pruned, test_loader, device)
_, test_q = evaluate(qmodel, test_loader, device)

# Profiling: modello potato W8A8
prof_q = profile_model(qmodel, dtype="int8", latency=False, size_mb=int8_model_size_mb(qparams, bias_q, len(qmodel.quantizers())))

save_result("teacher_combo_struct_int8", val_acc=val_q, test_acc=test_q, conv_ratio=CONV_RATIO, fc_ratio=FC_RATIO, skip_first=SKIP_FIRST, **prof_q)


c:\Users\Pietr\Desktop\Model-Compression\.venv\lib\site-packages\torch\utils\data\dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Tabella comparativa

In [13]:
names = [
    "teacher_baseline",
    "teacher_int8",
    "teacher_struct",
    "teacher_combo_struct",
    "teacher_combo_struct_int8"
]

rows = {
    n: json.load(open(f"../results/{n}.json"))
    for n in names
    if os.path.exists(f"../results/{n}.json")
}

df_teacher = (
    pd.DataFrame(rows).T
    .drop(columns=["name"], errors="ignore")
    .apply(pd.to_numeric, errors="coerce")
)

t = df_teacher.loc["teacher_baseline"]

table_teacher = pd.DataFrame({
    "test acc (%)": (df_teacher["test_acc"] * 100).round(2),
    "Δ vs baseline (pt)": (
        (df_teacher["test_acc"] - t["test_acc"]) * 100
    ).round(2),
    "params": df_teacher["params"].round().astype("Int64"),
    "size (MB)": df_teacher["size_mb"].round(3),
    "compression": (
        t["size_mb"] / df_teacher["size_mb"]
    ).round(2),
    "MACs (M)": (df_teacher["macs"] / 1e6).round(2),
    "peak RAM (KB)": df_teacher["peak_act_kb"].round(1),
    "energy (µJ)": df_teacher["energy_uj"].round(2),
    "energy reduction": (
        t["energy_uj"] / df_teacher["energy_uj"]
    ).round(2),
    "latency (ms)": df_teacher["latency_ms_median"].round(2)
})

display(table_teacher.style.format(precision=2, na_rep="–"))

,test acc (%),Δ vs baseline (pt),params,size (MB),compression,MACs (M),peak RAM (KB),energy (µJ),energy reduction,latency (ms)
teacher_baseline,89.28,0.00,15530762,62.16,1.00,215.69,327.70,992.17,1.00,8.30
teacher_int8,89.20,-0.08,15525258,15.57,3.99,215.69,81.90,64.71,15.33,–
teacher_struct,87.94,-1.34,10146885,40.59,1.53,210.31,327.70,967.44,1.03,7.22
teacher_combo_struct_int8,87.90,-1.38,10146885,10.17,6.11,210.31,81.90,63.09,15.73,–
